# Realistic-ish 3D manual: put the laptop on charge

The action-grounded 3D manual, now trying to **simulate reality** rather than draw boxes:
- **Rounded geometry** (superellipsoid "soft bricks") so the adapter, plug heads, strip and laptop have
  filleted edges like real plastic, not hard cubes.
- **Materials**: matte white plastic (adapter/heads), brushed metal (blades/pins/laptop), glossy black
  (cable), dark matte (strip), with smooth shading and a front-right key light.
- **Contact shadows** on the desk to ground the objects.

It is still hand-built low-ish poly - the point is it *tries* to look real. Uploading more photos of a
part is exactly how you would later replace a rough model with an accurate scanned/CAD one.

Layout and camera match the photo: strip vertical back-left, heads center, white adapter foreground,
laptop right, camera front-right and elevated. Press **Play**. Rotate/zoom freely. No GPU, no keys.

In [ ]:
!pip install -q plotly

In [ ]:
import math
import plotly.graph_objects as go

# ---------- vector helpers ----------
def sub(a,b): return (a[0]-b[0],a[1]-b[1],a[2]-b[2])
def add(a,b): return (a[0]+b[0],a[1]+b[1],a[2]+b[2])
def smul(a,s): return (a[0]*s,a[1]*s,a[2]*s)
def cross(a,b): return (a[1]*b[2]-a[2]*b[1],a[2]*b[0]-a[0]*b[2],a[0]*b[1]-a[1]*b[0])
def length(a): return math.sqrt(a[0]*a[0]+a[1]*a[1]+a[2]*a[2]) or 1e-9
def normalize(a): L=length(a); return (a[0]/L,a[1]/L,a[2]/L)
def lerp(a,b,t): return (a[0]+(b[0]-a[0])*t,a[1]+(b[1]-a[1])*t,a[2]+(b[2]-a[2])*t)
def perp(d):
    ref=(0.0,0.0,1.0) if abs(d[2])<0.9 else (1.0,0.0,0.0)
    a=normalize(cross(d,ref)); b=normalize(cross(d,a)); return a,b

# ---------- materials (Plotly lighting presets) ----------
MATTE =dict(ambient=0.50,diffuse=0.90,specular=0.12,roughness=0.92,fresnel=0.05)
SEMI  =dict(ambient=0.45,diffuse=0.85,specular=0.30,roughness=0.55,fresnel=0.15)
GLOSS =dict(ambient=0.38,diffuse=0.70,specular=0.65,roughness=0.22,fresnel=0.30)
METAL =dict(ambient=0.35,diffuse=0.48,specular=0.92,roughness=0.18,fresnel=0.55)
SHADOWMAT=dict(ambient=1.0,diffuse=0.0,specular=0.0)
LPOS=dict(x=250,y=-350,z=500)

# ---------- primitives -> (V, F, color, material, flat, opacity) ----------
def box_p(cx,cy,cz,sx,sy,sz,color,material=MATTE,flat=True):
    hx,hy,hz=sx/2.0,sy/2.0,sz/2.0
    V=[(cx-hx,cy-hy,cz-hz),(cx-hx,cy+hy,cz-hz),(cx+hx,cy+hy,cz-hz),(cx+hx,cy-hy,cz-hz),
       (cx-hx,cy-hy,cz+hz),(cx-hx,cy+hy,cz+hz),(cx+hx,cy+hy,cz+hz),(cx+hx,cy-hy,cz+hz)]
    F=[(0,1,2),(0,2,3),(4,6,5),(4,7,6),(0,4,5),(0,5,1),(1,5,6),(1,6,2),(2,6,7),(2,7,3),(3,7,4),(3,4,0)]
    return (V,F,color,material,flat,1.0)

def se_p(cx,cy,cz,ax,ay,az,e,color,material=MATTE,flat=False,nu=16,nv=10):
    # superellipsoid 'rounded brick' (e -> 0 boxier, e -> 1 ellipsoid)
    def cc(w,ex):
        co=math.cos(w); return math.copysign(abs(co)**ex,co)
    def ss(w,ex):
        si=math.sin(w); return math.copysign(abs(si)**ex,si)
    V=[]; row=nu+1
    for iv in range(nv+1):
        v=-math.pi/2+math.pi*iv/nv
        for iu in range(nu+1):
            u=-math.pi+2*math.pi*iu/nu
            V.append((cx+ax*cc(v,e)*cc(u,e), cy+ay*cc(v,e)*ss(u,e), cz+az*ss(v,e)))
    F=[]
    for iv in range(nv):
        for iu in range(nu):
            a=iv*row+iu; b=a+1; c=a+row; d=c+1
            F.append((a,b,c)); F.append((b,d,c))
    return (V,F,color,material,flat,1.0)

def cyl_p(p0,p1,r,color,material=METAL,flat=False,n=16):
    d=normalize(sub(p1,p0)); a,b=perp(d); V=[]
    for q in (p0,p1):
        for k in range(n):
            t=2*math.pi*k/n
            V.append(add(q,add(smul(a,r*math.cos(t)),smul(b,r*math.sin(t)))))
    c0=len(V); V.append(p0); c1=len(V); V.append(p1); F=[]
    for k in range(n):
        kk=(k+1)%n
        F.append((k,kk,n+k)); F.append((kk,n+kk,n+k)); F.append((c0,kk,k)); F.append((c1,n+k,n+kk))
    return (V,F,color,material,flat,1.0)

def tube_p(path,r,color,material=GLOSS,flat=False,n=10):
    V=[]; rings=[]
    for q in range(len(path)):
        prv=path[max(q-1,0)]; nxt=path[min(q+1,len(path)-1)]
        d=normalize(sub(nxt,prv)); a,b=perp(d); ring=[]
        for k in range(n):
            t=2*math.pi*k/n
            ring.append(len(V)); V.append(add(path[q],add(smul(a,r*math.cos(t)),smul(b,r*math.sin(t)))))
        rings.append(ring)
    F=[]
    for q in range(len(path)-1):
        for k in range(n):
            kk=(k+1)%n
            F.append((rings[q][k],rings[q][kk],rings[q+1][k]))
            F.append((rings[q][kk],rings[q+1][kk],rings[q+1][k]))
    return (V,F,color,material,flat,1.0)

def with_op(p,op): V,F,c,m,fl,_=p; return (V,F,c,m,fl,op)
def rot_x(p,ang,piv):
    V,F,c,m,fl,op=p; py,pz=piv[1],piv[2]; ca,sa=math.cos(ang),math.sin(ang)
    return ([(x,py+(y-py)*ca-(z-pz)*sa,pz+(y-py)*sa+(z-pz)*ca) for (x,y,z) in V],F,c,m,fl,op)

def mesh_one(p,tx):
    V,F,color,material,flat,op=p
    X=[v[0]+tx[0] for v in V]; Y=[v[1]+tx[1] for v in V]; Z=[v[2]+tx[2] for v in V]
    return go.Mesh3d(x=X,y=Y,z=Z,i=[f[0] for f in F],j=[f[1] for f in F],k=[f[2] for f in F],
                     color=color,opacity=op,flatshading=flat,lighting=material,lightposition=LPOS,
                     hoverinfo="skip")

def shadow(cx,cy,rx,ry):
    return mesh_one(with_op(se_p(cx,cy,0.012,rx,ry,0.012,1.0,"#23190d",SHADOWMAT,False,22,6),0.22),(0,0,0))

# ---------- colors ----------
WHITE="#f4f3ef"; OFF="#e9e7e2"; SILVER="#cfd3d9"; STEEL="#aeb3ba"; DARK="#26272b"; SLOT="#54555a"

# ---------- object geometry (local, centered at origin) ----------
def adapter_geo():
    p=[se_p(0,0,0, 0.47,0.49,0.37, 0.30, WHITE, MATTE, False, 18,11)]    # rounded white brick
    p.append(se_p(0,0.40,0.02, 0.30,0.10,0.24, 0.45, "#dcdbd6", SEMI, False,14,9))  # inlet plate (+y)
    p.append(box_p(-0.13,0.50,0.02, 0.05,0.06,0.15, "#2a2b2f", SEMI))   # inlet slot L
    p.append(box_p(0.13,0.50,0.02, 0.05,0.06,0.15, "#2a2b2f", SEMI))    # inlet slot R
    p.append(box_p(0,-0.49,-0.16, 0.26,0.06,0.10, "#2a2b2f", SEMI))     # magsafe port (-y)
    return p

def na_head_geo():
    p=[se_p(0,0,0, 0.19,0.17,0.19, 0.32, WHITE, MATTE, False, 12,9)]    # rounded body
    p.append(box_p(-0.08,0.30,0.0, 0.05,0.24,0.18, STEEL, METAL, False))# flat blade L
    p.append(box_p(0.08,0.30,0.0, 0.05,0.24,0.18, STEEL, METAL, False)) # flat blade R
    return p

def foreign_head_geo():
    p=[se_p(0,0,0, 0.19,0.17,0.19, 0.32, OFF, MATTE, False, 12,9)]
    p.append(cyl_p((-0.09,0.16,0),(-0.09,0.42,0), 0.045, "#9a9da2", METAL))  # round pin L
    p.append(cyl_p((0.09,0.16,0),(0.09,0.42,0), 0.045, "#9a9da2", METAL))    # round pin R
    return p

def strip_geo():
    p=[se_p(0,0,0, 0.26,0.30,1.32, 0.18, "#2c2c30", MATTE, False, 14,18)]  # rounded tall body
    p.append(box_p(0,-0.24,0, 0.36,0.10,2.36, "#1c1c1f", SEMI))            # recessed front
    for zz in (-0.95,-0.5,-0.05,0.4,0.85):
        p.append(box_p(0,-0.28,zz, 0.34,0.05,0.4, "#141416", SEMI))
        p.append(box_p(-0.07,-0.31,zz+0.07, 0.035,0.04,0.13, SLOT, SEMI))
        p.append(box_p(0.07,-0.31,zz+0.07, 0.035,0.04,0.13, SLOT, SEMI))
        p.append(box_p(0,-0.31,zz-0.1, 0.05,0.04,0.05, SLOT, SEMI))
    p.append(cyl_p((0,-0.30,-1.2),(0,-0.24,-1.2), 0.04, "#46e06a", GLOSS, False, 14))  # LED
    return p

def connector_geo():
    p=[se_p(0,0,0, 0.19,0.07,0.13, 0.35, SILVER, METAL, False, 12,8)]   # magnetic face
    p.append(box_p(0,-0.06,0, 0.26,0.03,0.05, "#2a2b2f", SEMI))         # contacts
    p.append(se_p(0,0.12,0, 0.07,0.09,0.07, 0.4, DARK, GLOSS, False,10,7))  # neck
    return p

def laptop_geo():
    p=[se_p(0,0,0, 0.9,0.62,0.06, 0.16, SILVER, METAL, False, 20,9)]    # rounded base slab
    p.append(box_p(0,0.08,0.065, 1.46,0.84,0.012, "#26282c", SEMI))     # keyboard well
    p.append(box_p(0,-0.4,0.067, 0.55,0.34,0.01, "#d6dadf", SEMI))      # trackpad
    p.append(box_p(-0.9,-0.05,0.03, 0.04,0.2,0.05, "#1c1c1f", SEMI))    # side charging port (-x)
    piv=(0,0.56,0.06)
    p.append(rot_x(se_p(0,0.56,0.62, 0.9,0.035,0.54, 0.16, SILVER, METAL, False,20,9), -0.32, piv))  # lid
    p.append(rot_x(box_p(0,0.50,0.62, 1.62,0.02,0.92, "#111317", SEMI), -0.32, piv))                 # display
    return p

DESK=box_p(0.0,2.2,-0.08, 9.0,6.0,0.12, "#a9824f", SEMI)
WALL=box_p(-0.5,4.95,1.6, 9.0,0.10,3.6, "#d6d4ca", MATTE)
ADAPTER,NAHEAD,FOREIGN=adapter_geo(),na_head_geo(),foreign_head_geo()
STRIP,CONNECTOR,LAPTOP=strip_geo(),connector_geo(),laptop_geo()

# ---------- poses (match the photo) ----------
ADAPTER_REST=(0.3,1.3,0.40); ADAPTER_PLUG=(-2.3,3.19,0.85)
NAHEAD_LOOSE=(-0.5,2.2,0.19); HEAD_OFFSET=(0.0,0.60,0.0)
FOREIGN_AT=(0.55,2.0,0.19)
STRIP_AT=(-2.3,4.5,1.3); LAPTOP_AT=(2.7,1.5,0.08)
CONN_REST=(-0.2,0.6,0.16); CONN_PLUG=(1.65,1.45,0.13)

def adapter_c(ph,t):
    if ph<1: return ADAPTER_REST
    if ph==1: return lerp(ADAPTER_REST,ADAPTER_PLUG,t)
    return ADAPTER_PLUG
def nahead_c(ph,t):
    if ph==0: return lerp(NAHEAD_LOOSE,add(ADAPTER_REST,HEAD_OFFSET),t)
    return add(adapter_c(ph,t),HEAD_OFFSET)
def conn_c(ph,t):
    if ph<2: return CONN_REST
    return lerp(CONN_REST,CONN_PLUG,t)
def cable_path(ph,t):
    a=adapter_c(ph,t); ap=(a[0],a[1]-0.5,a[2]-0.18)
    c=conn_c(ph,t);   cn=(c[0],c[1]+0.12,c[2])
    m1=lerp(ap,cn,0.33); m1=(m1[0],m1[1],0.12)
    m2=lerp(ap,cn,0.5);  m2=(m2[0]+0.35,m2[1]-0.1,0.10)
    m3=lerp(ap,cn,0.66); m3=(m3[0],m3[1],0.12)
    return [ap,m1,m2,m3,cn]

TITLES=[
 "Step 1 / 3   |   Attach the North-American flat-blade plug head to the adapter (leave the round-pin head).",
 "Step 2 / 3   |   Plug the adapter into an empty outlet on the power strip (the head rides along).",
 "Step 3 / 3   |   Connect the MagSafe connector to the laptop's side charging port  ->  charging.",
]
def target_pt(ph): return [(0.3,1.82,0.42),(-2.3,4.23,0.85),(1.8,1.45,0.16)][ph]
def take_pt(ph,t):
    if ph==0: return add(nahead_c(ph,t),(0,0,0.5))
    if ph==1: return add(adapter_c(ph,t),(0,0,0.7))
    return add(conn_c(ph,t),(0,0,0.45))

def label_here(ph):
    tp=target_pt(ph)
    return go.Scatter3d(x=[tp[0]],y=[tp[1]],z=[tp[2]],mode="markers+text",
        marker=dict(size=8,color="#e23b3b"),text=["HERE"],textposition="top center",
        textfont=dict(color="#e23b3b",size=13),hoverinfo="skip",showlegend=False)
def label_take(ph,t):
    kp=take_pt(ph,t)
    return go.Scatter3d(x=[kp[0]],y=[kp[1]],z=[kp[2]],mode="text",
        text=["TAKE"],textfont=dict(color="#ff8c1a",size=14),hoverinfo="skip",showlegend=False)

def static_traces():
    out=[mesh_one(DESK,(0,0,0)), mesh_one(WALL,(0,0,0))]
    out.append(shadow(LAPTOP_AT[0],LAPTOP_AT[1],1.05,0.72))
    out.append(shadow(FOREIGN_AT[0],FOREIGN_AT[1],0.28,0.24))
    for pr in STRIP: out.append(mesh_one(pr,STRIP_AT))
    for pr in LAPTOP: out.append(mesh_one(pr,LAPTOP_AT))
    for pr in FOREIGN: out.append(mesh_one(pr,FOREIGN_AT))
    return out

def moving_traces(ph,t):
    ac=adapter_c(ph,t); hc=nahead_c(ph,t); cc=conn_c(ph,t)
    out=[shadow(ac[0],ac[1],0.6,0.55), shadow(cc[0],cc[1],0.3,0.26)]
    for pr in ADAPTER: out.append(mesh_one(pr,ac))
    for pr in NAHEAD: out.append(mesh_one(pr,hc))
    for pr in CONNECTOR: out.append(mesh_one(pr,cc))
    out.append(mesh_one(tube_p(cable_path(ph,t),0.05,"#1d1d20",GLOSS,False,10),(0,0,0)))
    out.append(label_here(ph)); out.append(label_take(ph,t))
    return out

STATIC=static_traces()
M0=moving_traces(0,0.0)
MOVE_IDX=list(range(len(STATIC), len(STATIC)+len(M0)))

frames=[]; steps=[]; idx=0; PER=12
for ph in range(3):
    for f in range(PER+1):
        nm="f%d"%idx
        frames.append(go.Frame(data=moving_traces(ph,f/PER), traces=MOVE_IDX, name=nm,
                               layout=go.Layout(title=TITLES[ph])))
        steps.append(dict(args=[[nm],dict(frame=dict(duration=0,redraw=True),mode="immediate")],
                          label="",method="animate"))
        idx+=1

noax=dict(showbackground=False,showgrid=False,zeroline=False,showticklabels=False,title="")
fig=go.Figure(
    data=STATIC+M0,
    layout=go.Layout(
        title=TITLES[0], paper_bgcolor="#eef0f2",
        scene=dict(xaxis=dict(noax,range=[-3.4,4.4]),yaxis=dict(noax,range=[-0.6,5.4]),
                   zaxis=dict(noax,range=[-0.2,3.0]),aspectmode="data",bgcolor="#eef0f2",
                   camera=dict(eye=dict(x=1.75,y=-1.95,z=1.0),center=dict(x=-0.12,y=0.15,z=0.05))),
        width=980,height=700,
        updatemenus=[dict(type="buttons",showactive=False,x=0,y=1,
            buttons=[dict(label="Play",method="animate",
                          args=[None,dict(frame=dict(duration=60,redraw=True),fromcurrent=True)]),
                     dict(label="Pause",method="animate",
                          args=[[None],dict(frame=dict(duration=0,redraw=False),mode="immediate")])])],
        sliders=[dict(active=0,steps=steps,x=0.08,len=0.86,currentvalue=dict(visible=False))]),
    frames=frames)
fig.show()

## Notes
- **Rounded, shaded, material-tagged** parts try to read as real objects (matte white charger, metal
  blades, glossy cable, dark strip). Still hand-modelled and low-ish poly on purpose.
- The action logic is unchanged: source -> target, the head rides with the adapter, cable re-routes.
- **This is exactly where 'upload more photos' plugs in**: a rough hand-model here gets replaced by a
  reconstructed / retrieved model of *your* actual part, per object, on demand - the geometry improves
  where you look closer, the manual logic stays the same.
- Tune look: material presets `MATTE/SEMI/GLOSS/METAL`, light `LPOS`, camera `eye`/`center`.